# Data preprocessing

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely import make_valid

from andeangc import config as cfg
from andeangc import data_homogenize, data_update

## Peru [SENAMHI]

In [ ]:
folder = cfg.RESOURCES / "SENAMHI_PERU"
metadata, data = data_homogenize.process_excel_files(sorted(folder.glob("raw/*.xlsx")))

# metadata
metadata = metadata.groupby("gauge_name").agg(
    {column: (lambda operators: ", ".join(operators.unique())) if column == "operator" else "first"
     for column in metadata.columns if column != "gauge_name"}).reset_index()

metadata["gauge_id"] = data_homogenize.assign_gauge_ids(
    metadata["gauge_name"],
    cfg.RESOURCES / "gauge_ids_peru.csv",
    cfg.gauge_id_prefix_pe,
    cfg.gauge_id_zfill)
metadata = metadata.set_index("gauge_id")
metadata = metadata[~metadata.gauge_name.isin(cfg.senamhi_stations_to_remove)]
metadata.to_csv(folder / "SENAMHI_metadata.csv")

# data
data = data.loc[:, ~data.columns.duplicated(keep="first")]
data = data.loc[cfg.period_q[0]:cfg.period_q[1], metadata.gauge_name]
data.columns = metadata.index
data = data[metadata.index]
data.to_csv(folder / "SENAMHI_data.csv", index_label="date")

print(f"SENAMHI: {len(metadata)} gauges")

## Argentina [SNHI]

In [ ]:
folder = cfg.RESOURCES / "SNHI_ARG"

# data
data = data_homogenize.read_snhi_files(
    sorted(folder.glob("raw/*.xlsx")), cfg.gauge_id_prefix_ar, cfg.gauge_id_zfill)
data = data.loc[cfg.period_q[0]:cfg.period_q[1]]
data.to_csv(folder / "SNHI_data.csv", index_label="date")

# metadata
metadata = pd.read_csv(folder / "SNHI_metadata.csv")
metadata = data_homogenize.stamp_gauge_ids(metadata, cfg.gauge_id_prefix_ar, cfg.gauge_id_zfill)
metadata["gauge_name"] = metadata.river + " " + metadata.place
metadata.to_csv(folder / "SNHI_metadata.csv", index=False)

print(f"SNHI: {data.shape[1]} gauges")

## Chile [CAMELS-CL]

In [ ]:
folder = cfg.RESOURCES / "CAMELS_CL"

# data update (2020-2025)
data = data_update.update_camels_cl_data(
    folder / "CAMELS_CL_daily_1950_2020.csv",
    folder / "DGA_1960_2025.parquet",
    folder / "CAMELS_CL_daily_1950_2025.csv")

# metadata
metadata = pd.read_csv(folder / "CAMELS_CL_metadata.csv")
metadata = data_homogenize.stamp_gauge_ids(metadata, cfg.gauge_id_prefix_cl, cfg.gauge_id_zfill)
metadata.to_csv(folder / "CAMELS_CL_metadata.csv", index=False)

# basins
basins = gpd.read_file(folder / "basins_CAMELS_CL.gpkg")
basins = data_homogenize.stamp_gauge_ids(basins, cfg.gauge_id_prefix_cl, cfg.gauge_id_zfill)
basins["geometry"] = make_valid(basins.geometry)   # 208 of the 516 arrive invalid
basins.to_file(folder / "basins_CAMELS_CL.gpkg")
print(f"CAMELS-CL: {data.shape[1]} gauges")

## Patagonia [PMET-obs]

In [ ]:
folder = cfg.RESOURCES / "PMET_OBS"

# data update (2020-2025)
data = data_update.update_pmet_data(
    folder / "Q_PMETobs_1950_2020_v11d.csv",
    cfg.RESOURCES / "CAMELS_CL/DGA_1960_2025.parquet",
    cfg.RESOURCES / "SNHI_ARG/SNHI_data.csv",
    folder / "Q_PMETobs_1950_2025_v11d.csv")

# metadata
metadata = pd.read_csv(folder / "Q_PMETobs_v11_metadata.csv")
metadata = data_homogenize.stamp_gauge_ids(metadata, cfg.gauge_id_prefix_pa, cfg.gauge_id_zfill)
metadata.to_csv(folder / "Q_PMETobs_v11_metadata.csv", index=False)

# basins
basins = gpd.read_file(folder / "basins_PMETobs_v11.gpkg")
basins = data_homogenize.stamp_gauge_ids(basins, cfg.gauge_id_prefix_pa, cfg.gauge_id_zfill)
basins.to_file(folder / "basins_PMETobs_v11.gpkg")
print(f"PMET-obs: {data.shape[1]} gauges")